In [59]:
import pandas as pd 
df=pd.read_csv('Bank Customer Churn Prediction.csv')
df

,customer_id,credit_score,country,gender,age,tenure,balance,products_number,credit_card,active_member,estimated_salary,churn
0,15634602,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,15647311,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,15619304,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,15701354,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,15737888,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0
...,...,...,...,...,...,...,...,...,...,...,...,...
9995,15606229,771,France,Male,39,5,0.00,2,1,0,96270.64,0
9996,15569892,516,France,Male,35,10,57369.61,1,1,1,101699.77,0
9997,15584532,709,France,Female,36,7,0.00,1,0,1,42085.58,1
9998,15682355,772,Germany,Male,42,3,75075.31,2,1,0,92888.52,1


In [60]:
df=df.drop(['customer_id'],axis=1)
df.head()

,credit_score,country,gender,age,tenure,balance,products_number,credit_card,active_member,estimated_salary,churn
0,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [61]:
!pip install scikit-learn
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler

le=LabelEncoder()
df['gender']=le.fit_transform(df['gender'])
ohe=OneHotEncoder(sparse_output=False)
cols=ohe.fit_transform(df[['country']])
cols=pd.DataFrame(cols,columns=ohe.get_feature_names_out())
df=pd.concat([df,cols],axis=1)
df=df.drop(['country'],axis=1)
df.head()

,credit_score,gender,age,tenure,balance,products_number,credit_card,active_member,estimated_salary,churn,country_France,country_Germany,country_Spain
0,619,0,42,2,0.00,1,1,1,101348.88,1,1.0,0.0,0.0
1,608,0,41,1,83807.86,1,0,1,112542.58,0,0.0,0.0,1.0
2,502,0,42,8,159660.80,3,1,0,113931.57,1,1.0,0.0,0.0
3,699,0,39,1,0.00,2,0,0,93826.63,0,1.0,0.0,0.0
4,850,0,43,2,125510.82,1,1,1,79084.10,0,0.0,0.0,1.0


In [62]:
import pickle
with open('label_encoder_gender.pkl','wb') as file:
    pickle.dump(le, file)
with open('onehot_encodr_country.pkl','wb') as file:
    pickle.dump(ohe, file)

In [63]:
from sklearn.model_selection import train_test_split

X=df.drop(['churn'],axis=1)
y=df['churn']

In [64]:
X_train, X_test, y_train, y_test = train_test_split( X,y, train_size=0.2, random_state=42)

In [65]:
scaler=StandardScaler()
X_train=scaler.fit_transform(X_train)
X_test=scaler.transform(X_test)

In [66]:
with open('standard_scaler.pkl','wb') as file:
    pickle.dump(scaler,file)

# ANN Implementation

In [67]:
import tensorflow as tf 
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping, TensorBoard

In [68]:
#Build ANN model
model=Sequential([
    Dense(64, activation='relu', input_shape=(X_train.shape[1],)), #HL1
    Dense(32, activation='relu'), #HL2
    Dense(1, activation='sigmoid') #o/p
])

c:\Users\ASMITA\anaconda3\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [69]:
model.summary()
# param= weights and bias of that lower

Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_13 (Dense)                │ (None, 64)             │           832 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_14 (Dense)                │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_15 (Dense)                │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,945 (11.50 KB)

 Trainable params: 2,945 (11.50 KB)

 Non-trainable params: 0 (0.00 B)

In [70]:
#  Optimzers here adam  
opt=tf.keras.optimizers.Adam(learning_rate=0.01)
loss=tf.keras.losses.BinaryCrossentropy()

In [71]:
# compile model
model.compile(optimizer=opt, loss=loss, metrics =['accuracy'])
# or directly, model.compile(optmizer='adam', loss='binary_crossentropy', metric can be stacked])

In [72]:
import datetime 
log_dir='logs/fit/'+datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
tensorflow_callback=TensorBoard(log_dir=log_dir, histogram_freq=1)

In [73]:
# SEtup early stopping - why? say we run for 100 epochs but while we train we continuously monitor the loss
# value if it stops varying too much,
# we stop at that many epochs, we dont need to unnecessarily continue w too many epochs
early_stopping_callback=EarlyStopping(monitor='val_loss',patience=10, restore_best_weights=True)
#be patient for atleast 10 epochs, monitor loss value and restore the best weights

In [74]:
# TRain model now, uses validation data to compare 
history=model.fit(
    X_train, y_train, validation_data=(X_test, y_test), epochs=100, callbacks=[
        tensorflow_callback, early_stopping_callback
    ]
)

Epoch 1/100
63/63 ━━━━━━━━━━━━━━━━━━━━ 16s 66ms/step - accuracy: 0.8115 - loss: 0.4556 - val_accuracy: 0.8201 - val_loss: 0.4174
Epoch 2/100
63/63 ━━━━━━━━━━━━━━━━━━━━ 5s 67ms/step - accuracy: 0.8430 - loss: 0.3853 - val_accuracy: 0.8461 - val_loss: 0.3733
Epoch 3/100
63/63 ━━━━━━━━━━━━━━━━━━━━ 3s 30ms/step - accuracy: 0.8435 - loss: 0.3679 - val_accuracy: 0.8465 - val_loss: 0.3666
Epoch 4/100
63/63 ━━━━━━━━━━━━━━━━━━━━ 2s 34ms/step - accuracy: 0.8545 - loss: 0.3465 - val_accuracy: 0.8459 - val_loss: 0.3720
Epoch 5/100
63/63 ━━━━━━━━━━━━━━━━━━━━ 3s 39ms/step - accuracy: 0.8560 - loss: 0.3397 - val_accuracy: 0.8436 - val_loss: 0.3763
Epoch 6/100
63/63 ━━━━━━━━━━━━━━━━━━━━ 2s 31ms/step - accuracy: 0.8620 - loss: 0.3279 - val_accuracy: 0.8453 - val_loss: 0.3753
Epoch 7/100
63/63 ━━━━━━━━━━━━━━━━━━━━ 3s 39ms/step - accuracy: 0.8610 - loss: 0.3311 - val_accuracy: 0.8459 - val_loss: 0.3815
Epoch 8/100
63/63 ━━━━━━━━━━━━━━━━━━━━ 3s 48ms/step - accuracy: 0.8655 - loss: 0.3227 - val_accuracy: 0

In [75]:
model.save('model.h5')

In [76]:
# Load tensorboard extension
%load_ext tensorboard

The tensorboard extension is already loaded. To reload it, use:
  %reload_ext tensorboard


In [77]:
# Launch tensorboard session
%tensorboard --logdir logs/fit

Reusing TensorBoard on port 6006 (pid 23640), started 0:10:56 ago. (Use '!kill 23640' to kill it.)